# Clasificación de fraude en canastas de compra — FraudeCanastas.csv

## Objetivo

Construir un clasificador binario que identifique canastas de compra fraudulentas
(`fraud_flag`) a partir del dataset `FraudeCanastas.csv`, explorando y preparando los
datos, justificando la elección del algoritmo y sus hiperparámetros, y evaluando el
modelo resultante.

## Enfoque

De los algoritmos vistos en el módulo (Regresión Logística, Árbol de decisión / Random
Forest), se entrena una **Regresión Logística regularizada como baseline** interpretable
y un **Random Forest como modelo final**. La justificación completa de por qué Random
Forest es la elección final se desarrolla en la sección de Modelado, una vez que
conocemos bien la forma de los datos gracias al EDA.

## Nota sobre el dataset original

El enunciado del ejercicio referencia una liga externa con más información sobre el
dataset original (no es necesaria para resolver el ejercicio, que se evalúa sobre
`FraudeCanastas.csv`). Esa liga no se reproduce aquí; si la necesitas, consúltala en el
material del curso.


In [ ]:
# Librerías estándar disponibles en Google Colab (no se usan librerías externas)
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
# Validación cruzada reutilizada en todo el notebook (búsqueda de hiperparámetros,
# sensibilidad del filtro de frecuencia, y ajuste del umbral de decisión)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


## 1. Carga de datos

El archivo `FraudeCanastas.csv` viene comprimido dentro de `FraudeCanastas.zip`. Pandas
puede leer directamente un `.csv` dentro de un `.zip` sin necesidad de descomprimirlo
primero (detecta la compresión por la extensión del archivo), así que nunca dejamos el
CSV de ~92MB suelto en disco. **Para correr esta celda en Colab, sube `FraudeCanastas.zip`
al mismo directorio que este notebook** (o móntalo desde Google Drive y ajusta la ruta).


In [ ]:
DATA_PATH = "FraudeCanastas.zip"

# low_memory=False evita advertencias de tipos mixtos dado el altísimo número de columnas
df = pd.read_csv(DATA_PATH, low_memory=False)

# Verificación de forma esperada: si esto falla, algo cambió en el parseo del archivo
assert df.shape == (9319, 2457), f"Forma inesperada: {df.shape}"

df = df.set_index("ID")

TARGET_COL = "fraud_flag"
AGG_COLS = [
    "Nb_of_items",
    "total_of_items",
    "costo_total",
    "costo_medio_item",
    "costo_item_max",
    "costo_item_min",
]
# Las columnas de producto son todas las que no son ni agregados ni el target
PRODUCT_COLS = [c for c in df.columns if c not in AGG_COLS + [TARGET_COL]]

df[TARGET_COL] = df[TARGET_COL].astype(int)

print(f"Filas: {df.shape[0]:,} | Columnas totales: {df.shape[1]:,}")
print(f"Columnas de producto: {len(PRODUCT_COLS):,} | Columnas agregadas: {len(AGG_COLS)}")
df.head(3)


## 2. Exploración de datos (EDA)

### 2.1 Distribución del target

`fraud_flag` está claramente desbalanceado. Esto determina varias decisiones más
adelante: usar `stratify` en el split, `class_weight='balanced'` en los modelos, y
métricas como recall/precision/F1 de la clase minoritaria y PR-AUC en vez de accuracy.


In [ ]:
target_counts = df[TARGET_COL].value_counts().sort_index()
target_pct = df[TARGET_COL].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, ax=ax, palette=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["No fraude (0)", "Fraude (1)"])
ax.set_ylabel("Número de canastas")
ax.set_title("Distribución de fraud_flag")
for i, (count, pct) in enumerate(zip(target_counts.values, target_pct.values)):
    ax.text(i, count + 50, f"{count:,}\n({pct:.1f}%)", ha="center")
plt.tight_layout()
plt.show()


### 2.2 ¿El dataset viene ordenado?

Antes de hacer cualquier split, vale la pena revisar si las filas están en un orden
particular. Si el dataset viniera ordenado por el target, un split posicional (sin
`shuffle`) produciría conjuntos de entrenamiento/prueba completamente sesgados.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.scatter(range(len(df)), df[TARGET_COL].values, s=2, alpha=0.5)
ax.set_xlabel("Índice de fila (orden original del CSV)")
ax.set_ylabel("fraud_flag")
ax.set_title("fraud_flag a lo largo del archivo original")
plt.tight_layout()
plt.show()

# Contamos cuántas veces cambia el valor de fraud_flag al recorrer el archivo en orden
transitions = (df[TARGET_COL].values[1:] != df[TARGET_COL].values[:-1]).sum()
print(f"Número de transiciones en la secuencia original de fraud_flag: {transitions}")


**Hallazgo clave:** el archivo está ordenado — todas las filas de fraude (1) aparecen
primero, seguidas de todas las filas de no-fraude (0), con una sola transición en todo
el archivo. Esto confirma que es indispensable usar `shuffle=True` junto con
`stratify=y` al hacer el split (nunca un split posicional o secuencial).


### 2.3 Esparsidad de las columnas de producto

Cada columna de producto representa el monto gastado en un SKU específico dentro de la
canasta; la inmensa mayoría de las canastas no compran la mayoría de los SKUs, así que
se espera un dataset extremadamente disperso (muchos ceros).


In [ ]:
sparsity = (df[PRODUCT_COLS] == 0).values.mean()
print(f"Porcentaje de ceros en las columnas de producto: {sparsity:.3%}")


In [ ]:
# Frecuencia de aparición de cada producto (en cuántas canastas tiene un valor != 0)
product_frequency = (df[PRODUCT_COLS] != 0).sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(product_frequency.values, bins=50)
ax.set_yscale("log")
ax.set_xlabel("Número de canastas en las que aparece el producto")
ax.set_ylabel("Número de columnas de producto (escala log)")
ax.set_title("Distribución de frecuencia de aparición por producto")
plt.tight_layout()
plt.show()

print(f"Productos que aparecen en <=1 canasta: {(product_frequency <= 1).sum():,} de {len(PRODUCT_COLS):,}")
print(f"Productos que aparecen en <=5 canastas: {(product_frequency <= 5).sum():,} de {len(PRODUCT_COLS):,}")


Hay una cola larguísima de productos casi nunca comprados: la mayoría de las ~2,449
columnas de producto aportan poquísima señal individual y casi ningún caso para
aprender de ellas. Esto se resuelve en la sección de Preparación de datos con un filtro
de frecuencia mínima, en vez de intentar tratar caso por caso columnas "raras".


### 2.4 Variables agregadas vs. fraude

Comparamos la distribución de las 6 variables agregadas (`Nb_of_items`,
`total_of_items`, `costo_total`, `costo_medio_item`, `costo_item_max`,
`costo_item_min`) entre canastas fraudulentas y no fraudulentas.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, AGG_COLS):
    sns.boxplot(data=df, x=TARGET_COL, y=col, ax=ax, palette=["#4C72B0", "#C44E52"])
    ax.set_xticklabels(["No fraude", "Fraude"])
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


### 2.5 Correlación entre agregados y el target

Con solo 7 columnas (6 agregados + target) un heatmap de correlación es legible y
rápido de calcular. **No** se calcula un heatmap sobre las ~2,449 columnas de producto:
sería ilegible y computacionalmente innecesario para un dataset tan disperso.


In [ ]:
corr = df[AGG_COLS + [TARGET_COL]].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación entre agregados y fraud_flag")
plt.tight_layout()
plt.show()


### 2.6 Productos más frecuentes y productos más asociados al fraude

Aprovechando la esparsidad, calculamos directamente (sin matrices de correlación
completas): los 20 productos que más aparecen en general, y los 20 productos con mayor
diferencia en tasa de aparición entre canastas fraudulentas y no fraudulentas. Esto es
puramente descriptivo/exploratorio sobre el dataset completo; el filtro de columnas que
efectivamente se usará para modelar se calculará más adelante **solo sobre el conjunto
de entrenamiento**, para evitar fuga de información.


In [ ]:
top20_frequent = product_frequency.head(20)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top20_frequent.index[::-1], top20_frequent.values[::-1])
ax.set_xlabel("Número de canastas")
ax.set_title("Top 20 productos más frecuentes")
plt.tight_layout()
plt.show()


In [ ]:
rate_fraud = (df.loc[df[TARGET_COL] == 1, PRODUCT_COLS] != 0).mean()
rate_no_fraud = (df.loc[df[TARGET_COL] == 0, PRODUCT_COLS] != 0).mean()
rate_diff = (rate_fraud - rate_no_fraud).sort_values(key=lambda s: s.abs(), ascending=False)
top20_diff = rate_diff.head(20)

fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in top20_diff.values[::-1]]
ax.barh(top20_diff.index[::-1], top20_diff.values[::-1], color=colors)
ax.set_xlabel("Diferencia en tasa de aparición (fraude - no fraude)")
ax.set_title("Top 20 productos más asociados (o disociados) con el fraude")
plt.tight_layout()
plt.show()


### 2.7 Verificaciones de consistencia

Confirmamos dos supuestos sobre el esquema de datos antes de preparar las features:
que la suma de las columnas de producto por fila coincide con `costo_total`, y que no
hay valores nulos ni IDs duplicados.


In [ ]:
max_diff = (df[PRODUCT_COLS].sum(axis=1) - df["costo_total"]).abs().max()
print(f"Máxima diferencia entre suma(columnas_producto) y costo_total: {max_diff:.6f}")

print(f"Valores nulos totales: {df.isnull().sum().sum()}")
print(f"IDs duplicados: {df.index.duplicated().sum()}")


### 2.8 Resumen de hallazgos del EDA

- El target está desbalanceado (85.8% no fraude / 14.2% fraude) → hace falta
  `stratify`, `class_weight='balanced'` y métricas robustas al desbalance.
- El archivo viene **ordenado por el target** → el split debe usar `shuffle=True`.
- Las columnas de producto son extremadamente dispersas (>99% ceros) con una cola larga
  de SKUs casi nunca comprados → se necesita un filtro de frecuencia antes de modelar.
- Las columnas de producto son montos de costo (no conteos): su suma reconstruye
  exactamente `costo_total`.
- No hay nulos ni IDs duplicados que tratar.
- Algunos agregados (visibles en los boxplots) muestran diferencias de distribución
  entre fraude y no fraude, y algunos productos específicos aparecen con tasas muy
  distintas entre ambas clases — hay señal explotable tanto en agregados como en
  productos puntuales.


## 3. Preparación de datos

### 3.1 Split train/test

Hacemos el split **antes** de calcular cualquier estadístico derivado de los datos
(como el filtro de frecuencia de productos), para evitar fuga de información del
conjunto de prueba hacia el entrenamiento. Usamos `stratify=y` (por el desbalance) y
`shuffle=True` explícito (por el orden del archivo original, confirmado en el EDA).


In [ ]:
X = df[PRODUCT_COLS + AGG_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, shuffle=True, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("Proporción de fraude en train:", y_train.mean().round(4))
print("Proporción de fraude en test:", y_test.mean().round(4))


### 3.2 Filtro de frecuencia mínima para columnas de producto

En vez de tratar la columna con nombre "anómalo" (`APPLE PRODUCTDESCRIPTION | SAMSUNG |
MODEL90`, con un único valor distinto de cero en todo el dataset) como un caso especial,
la resolvemos de forma genérica: cualquier columna de producto que aparezca en muy pocas
canastas de **entrenamiento** aporta prácticamente nada al modelo y se descarta. El
umbral se calcula **solo sobre `X_train`** (nunca sobre test) para no filtrar con
información que el modelo no debería conocer en producción.


In [ ]:
freq_counts_train = (X_train[PRODUCT_COLS] != 0).sum()

thresholds = [1, 2, 5, 10, 20, 50]
retained = [(freq_counts_train > t).sum() for t in thresholds]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(thresholds, retained, marker="o")
ax.set_xlabel("Umbral mínimo de canastas (train) en que debe aparecer el producto")
ax.set_ylabel("Columnas de producto retenidas")
ax.set_title("Columnas retenidas vs. umbral de frecuencia")
for t, r in zip(thresholds, retained):
    ax.annotate(str(r), (t, r), textcoords="offset points", xytext=(0, 6), ha="center")
plt.tight_layout()
plt.show()


El conteo de columnas retenidas por sí solo no dice si el umbral elegido produce un
mejor o peor modelo. Para no elegir el punto de corte a ciegas, lo validamos con una
métrica de desempeño real: PR-AUC en validación cruzada dentro de train, para cada
umbral candidato, usando un Random Forest ligero (mismos hiperparámetros base que se
afinarán más adelante). Este resultado es la justificación empírica de `MIN_BASKETS`
que pide el enunciado, no solo un criterio de "menos columnas es mejor".


In [ ]:
def build_features(X_raw, keep_cols, drop_cols):
    """Conserva las columnas de producto frecuentes, agrega el resto en una sola
    columna de productos raros, y concatena los agregados."""
    X_freq = X_raw[keep_cols].copy()
    X_freq["costo_productos_raros"] = X_raw[drop_cols].sum(axis=1)
    return pd.concat([X_freq, X_raw[AGG_COLS]], axis=1)


probe_rf = RandomForestClassifier(
    n_estimators=300, max_depth=20, min_samples_leaf=1,
    class_weight="balanced", max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1,
)

sensitivity_rows = []
for min_baskets in [5, 10, 20, 50]:
    keep_cols = freq_counts_train[freq_counts_train > min_baskets].index.tolist()
    drop_cols = [c for c in PRODUCT_COLS if c not in keep_cols]
    X_probe = build_features(X_train, keep_cols, drop_cols)
    oof_proba = cross_val_predict(probe_rf, X_probe, y_train, cv=cv, method="predict_proba")[:, 1]
    sensitivity_rows.append({
        "MIN_BASKETS": min_baskets,
        "n_columnas_producto": len(keep_cols),
        "pr_auc_cv": average_precision_score(y_train, oof_proba),
    })

sensitivity_df = pd.DataFrame(sensitivity_rows).set_index("MIN_BASKETS").round(4)
sensitivity_df


`MIN_BASKETS=10` resulta ser el mejor punto entre los evaluados: retiene suficientes
columnas de producto para no perder señal (a diferencia de umbrales más agresivos como
20 o 50), pero descarta el ruido de la cola larga de SKUs casi únicos (a diferencia de
umbrales más laxos como 5). Se adopta con esta evidencia, no solo por el conteo de
columnas.


In [ ]:
MIN_BASKETS = 10  # confirmado empíricamente arriba: mejor PR-AUC (CV) entre los umbrales probados
frequent_cols = freq_counts_train[freq_counts_train > MIN_BASKETS].index.tolist()
rare_cols = [c for c in PRODUCT_COLS if c not in frequent_cols]

print(f"Columnas de producto frecuentes retenidas: {len(frequent_cols)}")
print(f"Columnas de producto raras agrupadas: {len(rare_cols)}")


Las columnas raras descartadas no se tiran del todo: se agregan en una sola columna
densa `costo_productos_raros` (suma de esas columnas por fila), para no perder por
completo la señal de la cola larga sin arrastrar miles de columnas casi vacías.


In [ ]:
X_train_final = build_features(X_train, frequent_cols, rare_cols)
X_test_final = build_features(X_test, frequent_cols, rare_cols)

print(f"Features finales de modelado: {X_train_final.shape[1]}")
X_train_final.head(3)


## 4. Modelado

### 4.1 Elección del algoritmo

De los dos algoritmos vistos en el módulo, la elección final es **Random Forest**,
usando **Regresión Logística regularizada como baseline** de comparación. Razones:

1. **Heterogeneidad de escalas**: tras el filtro de frecuencia, las features mezclan
   montos de costo (rango amplio) con conteos pequeños (`Nb_of_items`). Random Forest es
   invariante a la escala de las features; Regresión Logística necesita estandarizarlas
   (fuente extra de decisiones y de posibles errores).
2. **Interacciones no lineales**: es plausible que el fraude dependa de combinaciones
   (p. ej. `costo_item_max` alto junto con la presencia de ciertos productos
   específicos). Random Forest captura esas interacciones de forma nativa, sin tener que
   construir términos de interacción a mano como requeriría un modelo lineal.
3. **Interpretabilidad de negocio**: `feature_importances_` da un ranking directo de qué
   agregados y productos más pesan en la predicción — útil para un caso de uso
   antifraude donde el equipo de negocio quiere entender el "por qué".
4. Regresión Logística sigue siendo valiosa como **baseline rápido e interpretable vía
   coeficientes**, y sirve para confirmar que Random Forest efectivamente aporta valor
   sobre un modelo lineal simple, no solo complejidad injustificada.

### 4.2 Hiperparámetros

Para ambos modelos usamos `GridSearchCV` con grids acotados (para que el tiempo de
cómputo sea razonable tanto en esta validación local como en un runtime gratuito de
Colab), `StratifiedKFold(n_splits=3, shuffle=True)` explícito, y `scoring
='average_precision'` (equivalente a PR-AUC) porque es la métrica más alineada con un
problema desbalanceado como este (más informativa que accuracy o incluso ROC-AUC aquí).


In [ ]:
# --- Regresión Logística (baseline) ---
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced", solver="liblinear", max_iter=1000, random_state=RANDOM_STATE
    )),
])

lr_param_grid = {
    "clf__penalty": ["l1", "l2"],
    "clf__C": [0.01, 0.1, 1, 10],
}

lr_grid = GridSearchCV(
    lr_pipeline, lr_param_grid, scoring="average_precision", cv=cv, n_jobs=-1
)
lr_grid.fit(X_train_final, y_train)

print("Mejores hiperparámetros (LR):", lr_grid.best_params_)
print(f"Mejor PR-AUC en CV (LR): {lr_grid.best_score_:.4f}")


In [ ]:
# --- Random Forest (modelo final) ---
# n_jobs=1 en el estimador para no anidar paralelismo con GridSearchCV(n_jobs=-1)
rf = RandomForestClassifier(
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1
)

rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt", None],
}

rf_grid = GridSearchCV(
    rf, rf_param_grid, scoring="average_precision", cv=cv, n_jobs=-1
)
rf_grid.fit(X_train_final, y_train)

print("Mejores hiperparámetros (RF):", rf_grid.best_params_)
print(f"Mejor PR-AUC en CV (RF): {rf_grid.best_score_:.4f}")


## 5. Evaluación

Evaluamos ambos modelos sobre el conjunto de prueba (`X_test_final`, nunca visto
durante el entrenamiento ni la búsqueda de hiperparámetros). Dado el desbalance, el
foco está en precision/recall/F1 de la clase fraude, además de ROC-AUC y PR-AUC.


In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"=== {model_name} ===")
    print(classification_report(y_test, y_pred, target_names=["No fraude", "Fraude"]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No fraude", "Fraude"]).plot(
        ax=axes[0], cmap="Blues", colorbar=False
    )
    axes[0].set_title(f"Matriz de confusión — {model_name}")

    RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
    axes[1].set_title(f"Curva ROC — {model_name}")

    PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2])
    axes[2].set_title(f"Curva Precision-Recall — {model_name}")

    plt.tight_layout()
    plt.show()

    return {
        "model": model_name,
        "precision_fraud": precision_score(y_test, y_pred),
        "recall_fraud": recall_score(y_test, y_pred),
        "f1_fraud": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }


lr_metrics = evaluate_model(lr_grid.best_estimator_, X_test_final, y_test, "Regresión Logística")


In [ ]:
rf_metrics = evaluate_model(rf_grid.best_estimator_, X_test_final, y_test, "Random Forest")


### 5.1 Comparación de modelos


In [ ]:
comparison = pd.DataFrame([lr_metrics, rf_metrics]).set_index("model").round(4)
comparison


### 5.2 Importancia de variables (Random Forest)


In [ ]:
importances = pd.Series(
    rf_grid.best_estimator_.feature_importances_, index=X_train_final.columns
).sort_values(ascending=False)

top20_importances = importances.head(20)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top20_importances.index[::-1], top20_importances.values[::-1])
ax.set_xlabel("Importancia (Random Forest)")
ax.set_title("Top 20 features más importantes")
plt.tight_layout()
plt.show()


### 5.3 Ajuste del punto de operación (umbral de decisión)

`model.predict()` usa por defecto un umbral de probabilidad de 0.5, pero ese valor no
tiene nada de especial para este problema: en detección de fraude, el punto de corte
debería elegirse según cuánto le cuesta al negocio un falso negativo (fraude no
detectado) frente a un falso positivo (canasta legítima marcada para revisión). Como
ejercicio de ese ajuste, buscamos el umbral que maximiza F1 usando predicciones
out-of-fold **sobre train** (nunca sobre test, para no filtrar información), y lo
aplicamos como referencia sobre test.


In [ ]:
rf_oof_proba = cross_val_predict(
    rf_grid.best_estimator_, X_train_final, y_train, cv=cv, method="predict_proba"
)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_train, rf_oof_proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-12)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

print(f"Umbral óptimo (máximo F1, validación cruzada sobre train): {best_threshold:.3f}")
print(
    f"En esa validación: precision={precisions[best_idx]:.3f} "
    f"recall={recalls[best_idx]:.3f} f1={f1_scores[best_idx]:.3f}"
)


In [ ]:
rf_test_proba = rf_grid.best_estimator_.predict_proba(X_test_final)[:, 1]
y_pred_default = (rf_test_proba >= 0.5).astype(int)
y_pred_tuned = (rf_test_proba >= best_threshold).astype(int)

threshold_comparison = pd.DataFrame([
    {
        "umbral": 0.50,
        "precision_fraud": precision_score(y_test, y_pred_default),
        "recall_fraud": recall_score(y_test, y_pred_default),
        "f1_fraud": f1_score(y_test, y_pred_default),
    },
    {
        "umbral": round(float(best_threshold), 3),
        "precision_fraud": precision_score(y_test, y_pred_tuned),
        "recall_fraud": recall_score(y_test, y_pred_tuned),
        "f1_fraud": f1_score(y_test, y_pred_tuned),
    },
]).set_index("umbral").round(4)
threshold_comparison


El umbral por defecto favorece más el recall; el umbral ajustado sube algo la
precisión a cambio de recall. Ninguno de los dos es "el correcto" en abstracto — es una
decisión de negocio sobre qué error duele más. Para el resumen ejecutivo reportamos el
modelo con umbral por defecto (0.5), por ser el comportamiento estándar de
`predict()`, y dejamos esta tabla como evidencia de que el punto de operación es
ajustable según la tolerancia al riesgo del negocio.


### 5.4 Nota sobre fuga de datos

Como salvaguarda: si alguna métrica saliera sospechosamente perfecta (por ejemplo
ROC-AUC o recall muy cercanos a 1.0), sería señal de alerta para revisar que el filtro
de frecuencia se calculó únicamente sobre `X_train` y que ningún agregado esté
codificando trivialmente el target. Las métricas obtenidas abajo se revisan bajo ese
criterio antes de reportarlas en el resumen ejecutivo.


In [ ]:
final_metrics = {
    "lr": {k: round(v, 4) for k, v in lr_metrics.items() if k != "model"},
    "rf": {k: round(v, 4) for k, v in rf_metrics.items() if k != "model"},
    "rf_threshold_tuned": {
        "threshold": round(float(best_threshold), 3),
        "precision_fraud": round(precision_score(y_test, y_pred_tuned), 4),
        "recall_fraud": round(recall_score(y_test, y_pred_tuned), 4),
        "f1_fraud": round(f1_score(y_test, y_pred_tuned), 4),
    },
    "rf_best_params": rf_grid.best_params_,
    "lr_best_params": lr_grid.best_params_,
    "top5_features": top20_importances.head(5).index.tolist(),
    "n_features": int(X_train_final.shape[1]),
    "n_train": int(X_train_final.shape[0]),
    "n_test": int(X_test_final.shape[0]),
    "random_baseline_pr_auc": round(float(y_test.mean()), 4),
    "min_baskets_sensitivity": sensitivity_df.reset_index().to_dict(orient="records"),
}
print(json.dumps(final_metrics, indent=2, ensure_ascii=False))


## 6. Resumen ejecutivo

<!-- RESUMEN_EJECUTIVO_PLACEHOLDER -->
_Este resumen se completa automáticamente con los resultados reales después de ejecutar
el notebook de punta a punta._
